# 05 - PPO Training from Scratch (GPU Accelerated + Proactive Arrival Lookahead)

This notebook runs the 500,000 timestep production PPO training loop with:
1. **Proactive Future Arrival Lookahead** ($7$ global cluster & arrival hint features).
2. **Refined Multi-Objective Rewards** (turnaround efficiency ratio + anti-fragmentation bin-packing bonuses).
3. **Vectorized Domain Randomization** across all $6$ benchmark scenarios simultaneously ($N=8$ parallel CPU environments).
4. **Logit-level Action Masking** and **GPU-Accelerated CUDA Training** (`NVIDIA GeForce RTX 3050 Laptop GPU`).

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in python path
sys.path.insert(0, os.path.abspath('..'))

from rl.config import PPOConfig
from rl.trainer import PPOTrainer

print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active Device: {torch.cuda.get_device_name(0)}")

PyTorch: 2.7.1+cu118 | CUDA Available: True
Active Device: NVIDIA GeForce RTX 3050 Laptop GPU


## 1. Configure PPO Hyperparameters

In [2]:
config = PPOConfig(
    cluster_config="../configs/cluster_small.yaml",
    reward_config="../configs/reward.yaml",
    scenario="mixed",  # Parallel workers train across all 6 scenarios simultaneously
    num_envs=8,
    rollout_length=256,
    minibatch_size=128,
    epochs_per_update=10,
    total_timesteps=500000,  # 500,000 steps production run (~30 mins on RTX 3050)
    learning_rate=0.0003,
    device="cuda" if torch.cuda.is_available() else "cpu",
    checkpoint_dir="../checkpoints",
)

print("=== PPO Training Config ===")
for k, v in config.__dict__.items():
    print(f"  {k:22s}: {v}")

=== PPO Training Config ===
  cluster_config        : ../configs/cluster_small.yaml
  reward_config         : ../configs/reward.yaml
  scenario              : mixed
  max_queue_size        : 16
  sim_horizon_seconds   : 3600.0
  seed                  : 42
  learning_rate         : 0.0003
  gamma                 : 0.99
  gae_lambda            : 0.95
  clip_ratio            : 0.2
  entropy_coef          : 0.01
  value_coef            : 0.5
  max_grad_norm         : 0.5
  num_envs              : 8
  rollout_length        : 256
  minibatch_size        : 128
  epochs_per_update     : 10
  total_timesteps       : 500000
  hidden_dim            : 256
  device                : cuda
  checkpoint_dir        : ../checkpoints
  save_freq_steps       : 25000
  eval_freq_steps       : 10000
  eval_episodes         : 3


## 2. Execute Training Loop

In [ ]:
trainer = PPOTrainer(config)
history = trainer.train()

print("Training Complete!")

Starting PPO Training on [CUDA] with 8 Parallel Envs...
Total target timesteps: 500,000 (Batch size per update: 2048)

Step 010240/500000 | FPS: 203 | Mean Rew:  +54.61 | Loss(P): -0.0063 | Loss(V): 235.9541 | Entropy: 0.566 | KL: 0.0095
Step 020480/500000 | FPS: 234 | Mean Rew: +293.59 | Loss(P): -0.0113 | Loss(V): 142.6938 | Entropy: 0.581 | KL: 0.0098
Step 030720/500000 | FPS: 234 | Mean Rew: +302.39 | Loss(P): -0.0073 | Loss(V): 334.0289 | Entropy: 0.393 | KL: 0.0098


## 3. Training Telemetry & Loss Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 1. Mean Episode Reward
axes[0, 0].plot(history["timesteps"], history["mean_reward"], color='green', lw=2)
axes[0, 0].set_title("Mean Episode Reward")
axes[0, 0].set_xlabel("Timesteps")
axes[0, 0].grid(True, alpha=0.3)

# 2. Policy Loss (Clipped Surrogate)
axes[0, 1].plot(history["timesteps"], history["policy_loss"], color='crimson', lw=2)
axes[0, 1].set_title("Policy Loss (Clipped Surrogate)")
axes[0, 1].set_xlabel("Timesteps")
axes[0, 1].grid(True, alpha=0.3)

# 3. Value Function Loss (MSE)
axes[1, 0].plot(history["timesteps"], history["value_loss"], color='blue', lw=2)
axes[1, 0].set_title("Value Loss (Critic MSE)")
axes[1, 0].set_xlabel("Timesteps")
axes[1, 0].grid(True, alpha=0.3)

# 4. Policy Entropy
axes[1, 1].plot(history["timesteps"], history["entropy"], color='purple', lw=2)
axes[1, 1].set_title("Policy Entropy (Exploration)")
axes[1, 1].set_xlabel("Timesteps")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()